# Test 8.0 Phase 1 — HECKTOR PET format and loading

Explore how HECKTOR PET images look **before** adding them as nnUNet channel 1.

**Expected files (HECKTOR 2025 Task 1):**

| File | Role |
|------|------|
| `{case_id}__CT.nii.gz` | CT |
| `{case_id}__PT.nii.gz` | PET, **SUV** |
| `{case_id}.nii.gz` | Labels (GTVp=1, GTVn=2) |

Paths default to the same HECKTOR roots as Test5 (`TEST5_HECKTOR_TRAIN_SOURCE`, `TEST5_HECKTOR_TEST_SOURCE`).

This notebook does **not** build Dataset650 or train.

## 1. Imports and HECKTOR roots

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
import SimpleITK as sitk

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

REPO = Path.cwd().resolve()
if not (REPO / "image_processor").is_dir():
    for p in REPO.parents:
        if (p / "image_processor").is_dir():
            REPO = p
            break
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from image_processor.conventions import get_hecktor_paths
from image_processor.io.pet_align import (
    describe_nifti,
    hecktor_slices_to_use,
    resample_pet_to_ct,
    sitk_to_xyz,
)
from pipelines.test5.paths import default_hecktor_sources

SOURCES = default_hecktor_sources()
print("Repo:", REPO)
print("HECKTOR sources:")
for s in SOURCES:
    print(" ", s, "exists=" + str(s.is_dir()))

## 2. How to open PET (nibabel + SimpleITK)

- **nibabel** `get_fdata()`: array in NIfTI axis order (x, y, z). Good for slicing like the rest of this repo.
- **SimpleITK**: spacing / origin / direction. Use this to **resample PET onto CT** (linear). Do not numpy-reshape to the CT shape.
- Intensities on `__PT` are **SUV**. Do not run `NIfTIHandler.load_nii_image` (CT percentile → [0, 1]) when saving a training channel.

In [ ]:
def first_hecktor_case(sources):
    for src in sources:
        if not src.is_dir():
            continue
        for case_dir in sorted(src.iterdir()):
            if not case_dir.is_dir() or case_dir.name.startswith("."):
                continue
            paths = get_hecktor_paths(str(case_dir), case_dir.name)
            if os.path.isfile(paths["path_ct"]) and os.path.isfile(paths["path_mask"]):
                return case_dir, paths
    raise FileNotFoundError("No HECKTOR case with CT+mask under SOURCES")

case_dir, paths = first_hecktor_case(SOURCES)
print("Case:", case_dir.name)
for k, v in paths.items():
    print(f"  {k}: {v}  exists={os.path.isfile(v)}")

pet_nii = nib.load(paths["path_pet"]) if os.path.isfile(paths["path_pet"]) else None
ct_nii = nib.load(paths["path_ct"])
print("\nnibabel CT shape:", ct_nii.shape)
if pet_nii is not None:
    pet = pet_nii.get_fdata().astype(np.float32)
    print("nibabel PET shape:", pet_nii.shape)
    print("PET SUV min/mean/max:", float(np.nanmin(pet)), float(np.nanmean(pet)), float(np.nanmax(pet)))
else:
    print("PET missing for this case")

## 3. Geometry: CT vs PET

In [ ]:
print("CT", describe_nifti(paths["path_ct"]))
if os.path.isfile(paths["path_pet"]):
    print("PET", describe_nifti(paths["path_pet"]))
    pet_on_ct = resample_pet_to_ct(paths["path_pet"], paths["path_ct"])
    ct_img = sitk.ReadImage(paths["path_ct"])
    print("After resample, PET size == CT size?", pet_on_ct.GetSize() == ct_img.GetSize())
    print("Spacing match?", pet_on_ct.GetSpacing() == ct_img.GetSpacing())
else:
    print("Skip resample: no __PT file")

## 4. Axial overlay (CT + PET + GTVp/GTVn)

PET is shown after resample to CT. Slice index follows the Test5 tumor crop window (expansion=5).

In [ ]:
if not os.path.isfile(paths["path_pet"]):
    print("No PET — skip overlay")
else:
    ct_xyz = sitk_to_xyz(sitk.ReadImage(paths["path_ct"]))
    pet_xyz = sitk_to_xyz(resample_pet_to_ct(paths["path_pet"], paths["path_ct"]))
    mask = nib.load(paths["path_mask"]).get_fdata().astype(np.int32)
    slices = hecktor_slices_to_use(mask)
    z = slices[len(slices) // 2]

    ct_sl = np.rot90(ct_xyz[:, :, z])
    pet_sl = np.rot90(pet_xyz[:, :, z])
    m_sl = np.rot90(mask[:, :, z])
    lo, hi = np.percentile(ct_sl, (1, 99))
    ct_show = np.clip((ct_sl - lo) / (hi - lo + 1e-8), 0, 1)
    pet_vmax = np.percentile(pet_sl, 99) if np.isfinite(pet_sl).any() else 1.0

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(ct_show, cmap="gray")
    axes[0].set_title(f"CT z={z}")
    axes[1].imshow(pet_sl, cmap="inferno", vmin=0, vmax=max(pet_vmax, 1e-6))
    axes[1].set_title("PET (SUV, on CT grid)")
    axes[2].imshow(ct_show, cmap="gray")
    axes[2].imshow(np.ma.masked_where(m_sl != 1, m_sl), cmap="Reds", alpha=0.5, vmin=0, vmax=2)
    axes[2].set_title("CT + GTVp")
    axes[3].imshow(ct_show, cmap="gray")
    axes[3].imshow(np.ma.masked_where(m_sl != 2, m_sl), cmap="cool", alpha=0.5, vmin=0, vmax=2)
    axes[3].set_title("CT + GTVn")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 5. Inventory: missing `__PT` on train vs held-out test

In [ ]:
rows = []
for src in SOURCES:
    if not src.is_dir():
        print("Missing source (skipped):", src)
        continue
    n_ok = n_no_pet = n_no_ct = 0
    missing_pt = []
    for case_dir in sorted(src.iterdir()):
        if not case_dir.is_dir() or case_dir.name.startswith("."):
            continue
        p = get_hecktor_paths(str(case_dir), case_dir.name)
        has_ct = os.path.isfile(p["path_ct"])
        has_pt = os.path.isfile(p["path_pet"])
        has_mask = os.path.isfile(p["path_mask"])
        if not has_ct or not has_mask:
            n_no_ct += 1
            continue
        if not has_pt:
            n_no_pet += 1
            missing_pt.append(case_dir.name)
        else:
            n_ok += 1
    print(f"\n{src}")
    print(f"  CT+mask+PET: {n_ok}")
    print(f"  CT+mask, missing PET: {n_no_pet}")
    print(f"  skipped (no CT/mask): {n_no_ct}")
    if missing_pt[:20]:
        print("  first missing PET:", missing_pt[:20])
    rows.append((str(src), n_ok, n_no_pet))

print("\nSummary (ok, missing_pet):", rows)